# Tanzania — NASA POWER EDA (Week 0 Task 2)

Place the official extract at **`data/tanzania.csv`** (not committed). See the challenge brief for download instructions.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))

from eda_pipeline import (
    correlation_heatmap,
    handle_missing,
    initial_transform,
    load_raw,
    monthly_means_plot,
    monthly_precip_bars,
    replace_sentinels,
    repo_root_from_notebooks,
    top_correlations,
    zscore_outlier_mask,
)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")

COUNTRY_SLUG = "tanzania"
COUNTRY_NAME = "Tanzania"


## Data loading & date parsing


In [ ]:
ROOT = repo_root_from_notebooks()
df = load_raw(COUNTRY_SLUG, ROOT)
print("shape", df.shape, "columns:", list(df.columns))
df = initial_transform(df, COUNTRY_NAME)
df.head()


## NASA sentinel `-999`, duplicates

Replace `-999` with NaN **before** summary statistics ([NASA POWER missing values](https://power.larc.nasa.gov/docs/tutorials/quick-data-analysis-python/)).

Duplicate rows typically repeat identical values across **all** data columns.


In [ ]:
df = replace_sentinels(df)

ndup = int(df.duplicated().sum())
print("duplicate rows (full-row duplicates):", ndup)

if ndup > 0:
    dup_mask = df.duplicated(keep=False)
    cols_all = df.columns.tolist()
    print("duplicate investigation: rows share identical values across:", cols_all[:12], "...")
    display(df.loc[dup_mask].sort_values(by=cols_all).head(15))
    df = df.drop_duplicates().reset_index(drop=True)
    print("after drop_duplicates:", df.shape)
else:
    print("No duplicate rows.")

desc = df.describe()
display(desc)

na = df.isna().sum().sort_values(ascending=False)
pct = (na / len(df) * 100).round(2)
missing_report = pd.DataFrame({"count": na, "pct": pct}).loc[lambda x: x["count"] > 0]
missing_report = missing_report.sort_values("count", ascending=False)
display(missing_report)
high = missing_report[missing_report["pct"] > 5]
print("columns with >5% missing:", list(high.index) if len(high) else "none")


### Brief interpretation (`describe`)

- **T2M / T2M_MAX / T2M_MIN:** central tendency and spread for the single grid point.
- **PRECTOTCORR:** many dry-day zeros — mean can be small while wet days drive totals.
- **RH2M / QV2M / PS:** QC sanity (pressure should stay in a plausible surface range).


## Outlier screening — |Z| > 3

Applied to: **T2M**, **T2M_MAX**, **T2M_MIN**, **PRECTOTCORR**, **RH2M**, **WS2M**, **WS2M_MAX**.


In [ ]:
zcols = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "WS2M_MAX"]
mask = zscore_outlier_mask(df, zcols)
print("rows with |Z|>3 in any listed column:", int(mask.sum()))
if mask.any():
    display(df.loc[mask].head(10))


### Decision: retain, cap, or drop?

**Retain** flagged rows for this EDA. Extreme temperatures or rainfall often reflect real weather (heat waves, cloudbursts) that matter for COP-style narratives. Capping would hide tails; dropping would remove rare events. We only treat **missing** data below.


## Handle missing values

- Drop rows where **>30%** of the weather columns listed are missing.
- **Forward-fill** remaining gaps in time order for weather variables.


In [ ]:
weather_cols = [
    "T2M", "T2M_MAX", "T2M_MIN", "T2M_RANGE", "PRECTOTCORR",
    "RH2M", "WS2M", "WS2M_MAX", "PS", "QV2M",
]
df_clean, dropped_rows = handle_missing(df, weather_cols)
print("rows dropped (>30% missing in weather subset):", len(dropped_rows))
out_path = ROOT / "data" / f"{COUNTRY_SLUG}_clean.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(out_path, index=False)
print("exported:", out_path.resolve())


## Time series — monthly T2M & monthly total precipitation


In [ ]:
fig = monthly_means_plot(df_clean, COUNTRY_NAME)
plt.show()
fig2 = monthly_precip_bars(df_clean, COUNTRY_NAME)
plt.show()


### Trends / anomalies (commentary)

Scan the T2M line for multi-year drift vs interannual variability. Compare wet seasons via monthly PRECTOTCORR totals. This is exploratory — attribution needs multi-model ensembles and methods beyond raw point data.


## Correlations


In [ ]:
fig = correlation_heatmap(df_clean)
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(df_clean["T2M"], df_clean["RH2M"], s=4, alpha=0.35)
ax[0].set_xlabel("T2M (°C)")
ax[0].set_ylabel("RH2M (%)")
ax[0].set_title("T2M vs RH2M")
ax[1].scatter(df_clean["T2M_RANGE"], df_clean["WS2M"], s=4, alpha=0.35)
ax[1].set_xlabel("T2M_RANGE (°C)")
ax[1].set_ylabel("WS2M (m/s)")
ax[1].set_title("Diurnal range vs wind")
plt.tight_layout()
plt.show()

for a, b, r in top_correlations(df_clean, 3):
    print(f"{a} ↔ {b}: r = {r:.3f}")


### Three strongest correlations

Summarize the printed pairs: e.g. temperature variables co-move; humidity often drops as daily mean temperature rises in this sample.


## Distributions — rainfall histogram & bubble chart


In [ ]:
p = df_clean["PRECTOTCORR"].clip(lower=0)
skew = float(p.skew())

fig, ax = plt.subplots(figsize=(7, 4))
if skew > 2:
    ax.hist(np.log1p(p), bins=60, color="steelblue", edgecolor="white")
    ax.set_xlabel("log(1 + PRECTOTCORR)")
    ax.set_title(f"PRECTOTCORR — log1p scale (skew={skew:.2f})")
else:
    ax.hist(p, bins=60, color="steelblue", edgecolor="white")
    ax.set_xlabel("PRECTOTCORR (mm/day)")
plt.ylabel("count")
plt.tight_layout()
plt.show()

sample = df_clean.sample(min(2500, len(df_clean)), random_state=0)
sizes = np.clip(sample["PRECTOTCORR"] * 5, 1, 120)
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(
    sample["T2M"],
    sample["RH2M"],
    s=sizes,
    alpha=0.35,
    c=sample["Month"],
    cmap="viridis",
)
ax.set_xlabel("T2M (°C)")
ax.set_ylabel("RH2M (%)")
ax.set_title("Bubble size = PRECTOTCORR (mm/day)")
plt.colorbar(sc, ax=ax, label="Month")
plt.tight_layout()
plt.show()


## References used

- NASA POWER data access: https://power.larc.nasa.gov/
- WMO *State of the Climate in Africa*: https://wmo.int/publication-series/state-of-climate-africa-2024
- World Bank climate risk profiles: https://climateknowledgeportal.worldbank.org/
